# Raw Data Exploration

This notebook allows you to explore the raw data extracted by the ETL process (`artifacts/raw_users.parquet` and `artifacts/raw_insertions.parquet`).

We use **Polars** for efficient data handling.

In [ ]:
import polars as pl
import os

# Set display options
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(50)

## 1. Load Raw Data

In [ ]:
users_path = "../artifacts/raw_users.parquet"
insertions_path = "../artifacts/raw_insertions.parquet"

if os.path.exists(users_path):
    df_users = pl.read_parquet(users_path)
    print(f"Users: {df_users.shape}")
else:
    print("Users file not found.")

if os.path.exists(insertions_path):
    df_insertions = pl.read_parquet(insertions_path)
    print(f"Insertions: {df_insertions.shape}")
else:
    print("Insertions file not found.")

## 2. Explore Users

In [ ]:
if 'df_users' in locals():
    display(df_users.head())

## 3. Explore Insertions

In [ ]:
if 'df_insertions' in locals():
    display(df_insertions.head())

### 3.1 JSON Parsing Example
The `listing` and `selected_bundle` columns are stored as JSON strings. Here is how to parse them to extract specific fields.

In [ ]:
if 'df_insertions' in locals():
    # Example: Extract City, Price, and Title
    # Note: We use .str.json_decode() to parse the JSON string into a Struct, then access fields.
    
    df_parsed = df_insertions.select([
        pl.col("object_reference"),
        pl.col("submission_at"),
        
        # Extract City
        pl.col("listing_json").str.json_decode().struct.field("address").struct.field("locality").alias("city"),
        
        # Extract Rent Gross Price
        pl.col("listing_json").str.json_decode().struct.field("prices").struct.field("rent").struct.field("gross").alias("rent_gross"),
        
        # Extract Title (German)
        pl.col("listing_json").str.json_decode().struct.field("localization").struct.field("de").struct.field("text").struct.field("title").alias("title_de")
    ])
    
    display(df_parsed.head())

### 3.2 Check Customer Segments

In [ ]:
if 'df_insertions' in locals():
    print(df_insertions["customer_segment"].value_counts())